In [ ]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id=202320240SB101"

# -------------------------
# Fetch and clean bill text
# -------------------------
def fetch_text(url):
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return soup.get_text("\n")

# -------------------------
# Patterns
# -------------------------
amount_only_line_pattern = re.compile(r"^[\(\−\-]?\$?\d[\d,]*[\)]?$")

agency_header_pattern = re.compile(r"^[A-Z][A-Z0-9\/\-\s]+$")
skip_agency_headers = re.compile(r"^(CHAPTER|SECTION|ARTICLE)\b", re.IGNORECASE)

item_pattern = re.compile(r"^(\d{4}-\d{3}-\d{4})[—\-]\s*(.+)$")
program_pattern = re.compile(r"^(\d{4})-(.+?)(?:\.{2,}|…)?$", re.IGNORECASE)
object_pattern = re.compile(r"^(\d{6,7})-(.+?)(?:\.{2,}|…)?$", re.IGNORECASE)

schedule_header_pattern = re.compile(r"^Schedule:", re.IGNORECASE)
schedule_group_pattern = re.compile(r"^\(([\d\.]+)\)\s*$")

dotleader_trailing_amount = re.compile(
    r"(?:\.{2,}|…)\s*\$?([\d]{1,3}(?:,\d{3})+|\d+)\s*$"
)

# -------------------------
# Helpers
# -------------------------
def clean_amount(s: str) -> int:
    return int(
        s.replace(",", "")
         .replace("(", "-")
         .replace(")", "")
         .replace("−", "-")
         .replace("$", "")
         .strip()
    )

def extract_dotleader_amount(line: str):
    m = dotleader_trailing_amount.search(line)
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def find_amount_for_line(lines, i, max_lookahead=6):
    """
    Returns (amount, next_index_to_continue_from).
    Priority:
      1) dot-leader amount at end of current line
      2) amount on next line by itself
      3) dot-leader amount within next few lines
    """
    amt = extract_dotleader_amount(lines[i])
    if amt is not None:
        return amt, i + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        if amount_only_line_pattern.match(lines[j]):
            return clean_amount(lines[j]), j + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        amt2 = extract_dotleader_amount(lines[j])
        if amt2 is not None:
            return amt2, j + 1

    return None, i + 1

def extract_fund_name_from_item_title(item_title: str):
    m = re.search(r"payable\s+from\s+the\s+(.+)$", item_title, flags=re.IGNORECASE)
    return m.group(1).strip().rstrip(".") if m else None

# -------------------------
# Monetary-only parser (NO ACTIVITY)
# -------------------------
def parse_budget_money_only(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    current_agency = None
    current_item = None
    current_program_code = None
    current_fund_code = None
    current_fund_name = None

    in_schedule = False
    current_schedule_group = None

    rows = []
    i = 0

    while i < len(lines):
        line = lines[i]

        # Agency header
        if (
            agency_header_pattern.match(line)
            and len(line) > 8
            and not skip_agency_headers.match(line)
        ):
            current_agency = line.strip()
            i += 1
            continue

        # Schedule header
        if schedule_header_pattern.match(line):
            in_schedule = True
            current_schedule_group = None
            i += 1
            continue

        # Schedule group like "(2)" or "(.5)"
        m_sg = schedule_group_pattern.match(line)
        if m_sg and in_schedule:
            current_schedule_group = m_sg.group(1)
            i += 1
            continue

        # Item
        m_item = item_pattern.match(line)
        if m_item:
            current_item = m_item.group(1)
            item_title = m_item.group(2).strip()

            current_program_code = None
            in_schedule = False
            current_schedule_group = None

            current_fund_code = current_item.split("-")[-1]
            current_fund_name = extract_fund_name_from_item_title(item_title)

            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                rows.append({
                    "level": "item",
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": None,
                    "object_code": None,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": None,
                    "name": item_title,
                    "amount": amt
                })
                i = next_i
                continue

            i += 1
            continue

        # Ignore everything before first item
        if current_item is None:
            i += 1
            continue

        # Program
        m_prog = program_pattern.match(line)
        if m_prog:
            current_program_code = m_prog.group(1)
            prog_name = m_prog.group(2).strip()

            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                rows.append({
                    "level": "program",
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": current_program_code,
                    "object_code": None,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": current_schedule_group if in_schedule else None,
                    "name": prog_name,
                    "amount": amt
                })
                i = next_i
                continue

            i += 1
            continue

        # Object / Project
        m_obj = object_pattern.match(line)
        if m_obj:
            obj_code = m_obj.group(1)
            obj_name = m_obj.group(2).strip()

            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                program_code_for_row = current_program_code or obj_code[:4]

                rows.append({
                    "level": "object",
                    "agency": current_agency,
                    "item_number": current_item,
                    "program_code": program_code_for_row,
                    "object_code": obj_code,
                    "fund_code": current_fund_code,
                    "fund_name": current_fund_name,
                    "schedule_group": current_schedule_group if in_schedule else None,
                    "name": obj_name,
                    "amount": amt
                })

                i = next_i
                continue

            i += 1
            continue

        i += 1

    return rows

# -------------------------
# Run + save CSV
# -------------------------
def main():
    print("Fetching SB101 (2023) text...")
    text = fetch_text(URL)

    print("Parsing SB101 (2023) — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "CA_Budget_SB101_2023_money_only.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()


Fetching SB101 (2023) text...
Parsing SB101 (2023) — MONEY ONLY...

Sample rows:
  level                         agency   item_number program_code object_code fund_code                                            fund_name schedule_group                                                                                                  name     amount
   item LEGISLATIVE/JUDICIAL/EXECUTIVE 0110-001-0001         None        None      0001                                                 None           None                                                                                 For support of Senate  177325000
program LEGISLATIVE/JUDICIAL/EXECUTIVE 0110-001-0001         0960        None      0001                                                 None              1                                                                                 Support of the Senate  177325000
 object LEGISLATIVE/JUDICIAL/EXECUTIVE 0110-001-0001         0960      101001      0001                         

In [21]:
df = pd.read_csv(
    "CA_Budget_SB101_2023_money_only.csv",
    dtype={"program_code": "string", "object_code": "string", "fund_code": "string", "schedule_group": "string"}
)

In [20]:
df.head(200)

,level,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount
0,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,<NA>,0001,NaN,<NA>,For support of Senate,177325000
1,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,0960,<NA>,0001,NaN,1,Support of the Senate,177325000
2,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,0960,101001,0001,NaN,1,Salaries of Senators,-6751000
3,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,0960,317295,0001,NaN,1,Mileage,-11000
4,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,0960,317292,0001,NaN,1,Expenses,-1712000
...,...,...,...,...,...,...,...,...,...,...
195,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-031-0001,<NA>,<NA>,0001,NaN,<NA>,For support of Secretary of Transportation,500000
196,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-101-0046,<NA>,<NA>,0046,"Public Transportation Account, State Transport...",<NA>,"For Local Assistance, Secretary of Transportat...",280000000
197,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-101-0046,0277,<NA>,0046,"Public Transportation Account, State Transport...",1,Statewide Transportation Priorities,280000000
198,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-101-0890,<NA>,<NA>,0890,Federal Trust Fund,<NA>,"For local assistance, Secretary of Transportat...",66801000
